In [1]:
import torch
import torchvision
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf

In [2]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)

    except RuntimeError as e:
        print(e)

In [8]:
import os

print("Current directory: - collaborative_filtering.ipynb:3", os.getcwd())
print("Files/folders here: - collaborative_filtering.ipynb:4", os.listdir())

Current directory: - collaborative_filtering.ipynb:3 c:\Users\DELL\Desktop\AML 1st\exp5
Files/folders here: - collaborative_filtering.ipynb:4 ['collaborative_filtering.ipynb', 'movies.csv', 'movie_reviews', 'ratings.csv']


In [9]:
ratings_data = pd.read_csv('ratings.csv')
movie_names_data = pd.read_csv('movies.csv')

In [10]:
import sys
print(sys.executable)

c:\Users\DELL\Desktop\AML 1st\.venv\Scripts\python.exe


In [11]:
import tensorflow as tf
print(tf.__version__)

2.21.0


In [12]:
import tensorflow as tf
import os

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

gpus = tf.config.experimental.list_physical_devices('GPU')

if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU detected and configured - collaborative_filtering.ipynb:12")
    except RuntimeError as e:
        print(e)
else:
    print("No GPU detected, using CPU - collaborative_filtering.ipynb:16")

No GPU detected, using CPU - collaborative_filtering.ipynb:16


In [13]:
import pandas as pd

In [14]:
import os
print(os.getcwd())

c:\Users\DELL\Desktop\AML 1st\exp5


In [16]:
n_movies = len(movie_names_data)
n_user = len(ratings_data['userId'].unique())

In [17]:
ratings_data = pd.merge(ratings_data, movie_names_data, on='movieId', how='inner')

In [18]:
ratings_data.head()


,userId,movieId,rating,timestamp,title,genres
0,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,1,3,4.0,964981247,Grumpier Old Men (1995),Comedy|Romance
2,1,6,4.0,964982224,Heat (1995),Action|Crime|Thriller
3,1,47,5.0,964983815,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,5.0,964982931,"Usual Suspects, The (1995)",Crime|Mystery|Thriller


In [19]:
from sklearn.preprocessing import LabelEncoder
import random
Y = ratings_data.rating
user_enc = LabelEncoder()
movie_enc = LabelEncoder()
X = np.array([user_enc.fit_transform(ratings_data.userId),
              movie_enc.fit_transform(ratings_data.title)]).T

In [20]:
user_enc.classes_[4], movie_enc.classes_[8871]


(np.int64(5), 'Toy Story (1995)')

In [21]:
for x, y in zip(X[:10], Y[:10]):
    print(list(x), y)

[np.int64(0), np.int64(8871)] 4.0
[np.int64(0), np.int64(3661)] 4.0
[np.int64(0), np.int64(3845)] 4.0
[np.int64(0), np.int64(7523)] 5.0
[np.int64(0), np.int64(9119)] 5.0
[np.int64(0), np.int64(3252)] 3.0
[np.int64(0), np.int64(1284)] 5.0
[np.int64(0), np.int64(1337)] 4.0
[np.int64(0), np.int64(7180)] 5.0
[np.int64(0), np.int64(1535)] 5.0


In [22]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=0)

In [23]:
num_users = len(X)
num_movies = len(X)

In [24]:
from keras.layers import Input, Embedding, Flatten, Dot, Dense, Activation, Dropout
from keras.models import Model

def build_model():
    movie_input = Input(shape=[1], name="Book-Input")
    movie_embedding = Embedding(n_movies+1, 15, name="Book-Embedding")(movie_input)
    movie_vec = Flatten(name="Flatten-Books")(movie_embedding)

    user_input = Input(shape=[1], name="User-Input")
    user_embedding = Embedding(n_user+1, 15, name="User-Embedding")(user_input)
    user_vec = Flatten(name="Flatten-Users")(user_embedding)
    
    prod = Dot(name="Dot-Product", axes=1)([user_vec, movie_vec])
    
    prod = Dense(32)(prod)
    prod = Activation('relu')(prod)
    prod = Dropout(0.5)(prod)

    prod = Dense(16)(prod)
    prod = Activation('relu')(prod)
    prod = Dropout(0.5)(prod)
    prod = Dense(1)(prod)


    model = Model([user_input, movie_input], prod)
    model.compile('adam', 'mean_squared_error', metrics=['accuracy'])

    return model


model = build_model()

In [26]:
model_checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath='./checkpoint.weights.h5',
    save_weights_only=True,
    monitor='val_loss',
    mode='min',
    save_best_only=True,
    verbose=1
)

history = model.fit(
    [X_train[:, 0], X_train[:, 1]],
    Y_train,
    epochs=15,
    verbose=1,
    batch_size=64,
    validation_data=([X_test[:, 0], X_test[:, 1]], Y_test),
    callbacks=[model_checkpoint_callback]
)

Epoch 1/15
1247/1261 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.0224 - loss: 5.0126
Epoch 1: val_loss improved from None to 1.19562, saving model to ./checkpoint.weights.h5

Epoch 1: finished saving model to ./checkpoint.weights.h5
1261/1261 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.0266 - loss: 3.0009 - val_accuracy: 0.0280 - val_loss: 1.1956
Epoch 2/15
1251/1261 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.0281 - loss: 1.7309
Epoch 2: val_loss improved from 1.19562 to 0.97606, saving model to ./checkpoint.weights.h5

Epoch 2: finished saving model to ./checkpoint.weights.h5
1261/1261 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - accuracy: 0.0279 - loss: 1.6243 - val_accuracy: 0.0280 - val_loss: 0.9761
Epoch 3/15
1248/1261 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.0285 - loss: 1.1944
Epoch 3: val_loss improved from 0.97606 to 0.90365, saving model to ./checkpoint.weights.h5

Epoch 3: finished saving model to ./checkpoint.weights.h5
1261/1261 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - acc

In [27]:
X_test[:5], Y_test[:5]

(array([[ 275, 4337],
        [ 598, 7425],
        [ 482,  334],
        [ 201, 3548],
        [ 273, 3540]]),
 41008    5.0
 94274    2.5
 77380    2.5
 29744    3.0
 40462    4.0
 Name: rating, dtype: float64)

In [28]:
predictions = model.predict([X_test[:5, 0], X_test[:5, 1]])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 155ms/step


In [29]:
print(predictions,"\n\n - collaborative_filtering.ipynb:1", Y_test[:5].values)


[[4.3161325]
 [3.1232398]
 [3.8640337]
 [3.9874651]
 [3.2170625]] 

 - collaborative_filtering.ipynb:1 [5.  2.5 2.5 3.  4. ]


In [30]:
movie_enc.classes_[4]

"'Til There Was You (1997)"

In [38]:
extract_true_ratings(test_user_id, X_test)

[np.float64(4.0),
 np.float64(3.0),
 np.float64(4.0),
 np.float64(1.0),
 np.float64(5.0),
 np.float64(5.0)]

In [32]:
def extract_true_ratings(user_id, X_test):
    
    true_ratings = list()
    for x, y in X_test:
        if x == user_id:
            rating = ratings_data[(ratings_data['userId'] == user_enc.classes_[user_id]) \
                & (ratings_data['title'] == movie_enc.classes_[y])]['rating'].values[0]
            true_ratings.append(rating)

    return true_ratings

In [33]:
def predict_ratings(user_id, X_test):
    '''
    given user id predict all ratings for movies
    '''
    user_data = ratings_data[ratings_data['userId'] == user_id]
    movie_ids, movie_names, predictions, movie_genres = list(), list(), list(), list()
    i = 0
    for _id, movie_id in X_test:
        if user_id == X_test[i][0]:
            movie_ids.append(X_test[i, 1])
            movie_names.append(movie_enc.classes_[movie_id])
            pred = model.predict([ np.array([X_test[i, 0]]), np.array([X_test[i, 1]]) ])
            predictions.append(pred[0][0])
        i += 1
    return movie_ids, movie_names, movie_genres, predictions

In [34]:
test_user_id = 7
userid_rating_data = ratings_data[ratings_data['userId'] == test_user_id]
# userid_rating_data

In [35]:
movie_ids, movie_names, movie_genres, predictions = predict_ratings(test_user_id, X_test)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 127ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step


In [36]:
dictionary = {"user_id": [test_user_id]*len(movie_ids),
              "movie_id": movie_ids,
              "movie_name":movie_names,
              "predicted_ratings":predictions,
              "true_ratings": extract_true_ratings(test_user_id, X_test) 
              }

In [37]:
prediction_dataframe = pd.DataFrame.from_dict(dictionary, orient='index').transpose()
prediction_dataframe.sort_values('predicted_ratings', ascending=False)


,user_id,movie_id,movie_name,predicted_ratings,true_ratings
5,7,7421,Schindler's List (1993),3.922895,5.0
0,7,6865,Pulp Fiction (1994),3.922596,4.0
4,7,2139,Dances with Wolves (1990),3.766615,5.0
2,7,7912,Speed (1994),3.165221,4.0
3,7,1799,City Slickers II: The Legend of Curly's Gold (...,3.101639,1.0
1,7,7755,Sleepless in Seattle (1993),2.324693,3.0
